# Extreme Year Analysis: Cross-Basin Comparison & Final Ordering

##### Compares Pearson-III results (sample skew vs weighted skew), validates rankings across Russian River basins, and produces the final extreme year ordering for USACE HMS calibration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import sys
sys.path.insert(0, '../..')
from pathlib import Path
from UCB_training.UCB_utils import clean_df
from UCB_training.UCB_plotting import plot_availability, plot_flow_precip, SPLIT_COLORS, plot_basin_extremeness, plot_basin_flow_precip_extremeness

PLOT_DIR = Path('extreme_year_analysis/plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df_sample = pd.read_csv('extreme_year_analysis/data/peak_flow_analysis_sample_skew.csv')
df_weighted = pd.read_csv('extreme_year_analysis/data/peak_flow_analysis_weighted_skew.csv')

df_sample.set_index('water_year', inplace=True)
df_weighted.set_index('water_year', inplace=True)

BASINS = ['Calpella', 'Guerneville', 'Hopland', 'Warm Springs']

print(f"Years: {df_sample.index.min()}–{df_sample.index.max()} ({len(df_sample)} total)")
df_sample - df_weighted

##### Almost identical, go with no skew based on latest talk to Matt.

In [ ]:
df_sample

In [ ]:
ranks = df_sample.rank(ascending=True).astype(int)
ranks['mean_rank'] = ranks[BASINS].mean(axis=1)
ranks.sort_values('mean_rank', ascending=True, inplace=True)

print("Per-Basin Ranks (1=wettest, 15=driest)")
ranks

In [ ]:
n = 3
print(f"Top {n} Wettest (lowest exceedance = biggest floods) per Basin")
for b in BASINS:
    top_wet = ranks.nsmallest(n, b).index.tolist()
    print(f"{b:15s}: {top_wet}")

consensus_wet = set.intersection(*[set(ranks.nsmallest(n, b).index) for b in BASINS])
print(f"\nConsensus wet (all 4 basins agree): {sorted(consensus_wet) if consensus_wet else 'NONE'}")

print(f"\nTop {n} Driest (highest exceedance = smallest peaks) per Basin")
for b in BASINS:
    top_dry = ranks.nlargest(n, b).index.tolist()
    print(f"{b:15s}: {top_dry}")

##### Wettest: [1995, 2006] in all 4 basins top 3 (and usually #1 or #2). 1997 vs 2003 can be debatable. 1995 and 2006 top 2 wet years?
##### Driest: [2001] in all, #1 for 3 basins. [2009, 2007] in 3 basins. 

In [ ]:
BASIN_TARGET_COLS = {
    'Calpella': 'NR CALPELLA FLOW COE CPL',
    'Hopland': 'NR HOPLAND FLOW COE HOP',
    'Guerneville': 'NR GUERNEVILLE FLOW COE GRN',
    'Warm Springs': 'LAKE SONOMA FLOW-RES IN CALC-VAL-SHIFT-SMOOTH',
}

BASIN_PRECIP_COLS = {
    'Calpella': ['EF RUSSIAN 20 PRECIP-INC SCREENED'],
    'Hopland': ['RUSSIAN 60 PRECIP-INC SCREENED', 'RUSSIAN 70 PRECIP-INC SCREENED', 'WF RUSSIAN PRECIP-INC SCREENED'],
    'Warm Springs': ['DRY CREEK 20 PRECIP-INC SCREENED', 'DRY CREEK 30 PRECIP-INC SCREENED'],
    'Guerneville': ['BIG SULPHUR CR PRECIP-INC SCREENED', 'DRY CREEK 10 PRECIP-INC SCREENED',
                    'EF RUSSIAN 20 PRECIP-INC SCREENED', 'GREEN VALLEY PRECIP-INC SCREENED',
                    'LAGUNA PRECIP-INC SCREENED', 'RUSSIAN 20 PRECIP-INC SCREENED',
                    'RUSSIAN 30 PRECIP-INC SCREENED', 'RUSSIAN 40 PRECIP-INC SCREENED',
                    'RUSSIAN 50 PRECIP-INC SCREENED', 'RUSSIAN 60 PRECIP-INC SCREENED',
                    'RUSSIAN 70 PRECIP-INC SCREENED', 'SANTA ROSA CR 10 PRECIP-INC SCREENED',
                    'SANTA ROSA CR 20 PRECIP-INC SCREENED', 'WF RUSSIAN PRECIP-INC SCREENED'],
}

DATA_DIR = '../../russian_river_data'
df_all = clean_df(pd.read_csv(f'{DATA_DIR}/daily.csv', low_memory=False))
basin_data = {}

for basin, flow_col in BASIN_TARGET_COLS.items():
    precip_cols = BASIN_PRECIP_COLS[basin]
    mean_precip = df_all[precip_cols].mean(axis=1)

    bdf = pd.DataFrame({'flow': df_all[flow_col], 'precip': mean_precip}, index=df_all.index)
    bdf.index.name = 'date'
    bdf['water_year'] = np.where(bdf.index.month >= 10, bdf.index.year + 1, bdf.index.year)

    nans = bdf['flow'].isna().sum()
    basin_data[basin] = bdf
    print(f"{basin}: {len(bdf)} days, flow='{flow_col}', NaN={nans} ({nans/len(bdf)*100:.1f}%), precip=mean of {len(precip_cols)} cols")

SPLITS = {
    'Train': ('1994-10-01', '2002-09-30'),
    'Validation': ('2002-10-01', '2005-09-30'),
    'Test': ('2005-10-01', '2009-09-30')}

print(f"\nSplits: Train WY1995-2002 | Val WY2003-2005 | Test WY2006-2009")

In [ ]:
plot_availability(basin_data, SPLITS, save_path=PLOT_DIR / 'data_availability_daily.png')

##### Out of the years that ranked high on wetness/dryness, we have **1995** for all basins, **2006** for Hopland but seems fine and ranked top for all, **2007** for Calpella, seems dry / top 3 in all basins, AND more significantly **2009** for whch we have a significant chunk missing for 3 basins


## Deep Dive: Extreme Years with Missing Data

Years that ranked in the top-3 wettest or driest **and** have significant NaN gaps need verification. If the annual peak flow (which drives the Pearson-III ranking) falls within a gap, the ranking is unreliable.

| Water Year |          Ranking          | Concern |
|:---:|:-------------------------:|:---|
| **1995** |        #1 wettest         | ~226 days NaN for Calpella, Hopland, Guerneville (Oct–Mar, then Jul–Sep). Peak likely Jan 1995. |
| **2006** |        #2 wettest         | Hopland has a gap starting ~Jun 2006, but peak should be in winter. Low risk. |
| **2007** | Top-3 driest (3/4 basins) | Calpella has ~240 days NaN (Apr–Dec 2007). Peak observed before gap. |
| **2009** | Top-3 driest (3/4 basins) | ~216 days NaN for 3 basins starting ~Feb 26 2009. Peak may fall near/in gap. |

Zoomed flow + precip plots below confirm whether the annual peaks are actually observed.

In [ ]:
plot_flow_precip(basin_data, SPLITS, water_years=[1995], save_path=PLOT_DIR / 'flow_precip_WY1995.png')

**WY 1995**: Despite large NaN blocks (Oct–Dec 1994 and Jul–Sep 1995), the Jan 1995 peak, the largest flood event in the record, is clearly observed across all 4 basins. The missing data falls in dryer seasons and does not affect peak 7 day flow? I don't see a reason the Pearson-III ranking is invalidated. 

In [ ]:
plot_flow_precip(basin_data, SPLITS, water_years=[2006], save_path=PLOT_DIR / 'flow_precip_WY2006.png')

**WY 2006**: Clean across all basins. The winter flood peak is fully captured. Hopland's gap iw well before the peak. The other basins show similar flow and precip patterns.

In [ ]:
plot_flow_precip(basin_data, SPLITS, water_years=[2007], save_path=PLOT_DIR / 'flow_precip_WY2007.png')

**WY 2007**: Calpella loses data from ~Apr 2007 onward, but this is a **dry year** — the small winter peak is fully observed before the gap. The ranking is based on peak flow magnitude, which is captured. All other basins are complete or nearly so. For a baseflow/drought analysis, the missing data would be more concerning.

In [ ]:
plot_flow_precip(basin_data, SPLITS, water_years=[2009], save_path=PLOT_DIR / 'flow_precip_WY2009.png')

**WY 2009**: The most problematic year. Data cuts out ~Feb 26 for Calpella, Hopland, and Guerneville. Warm Springs is complete and does not show a concerning higher peak. The **drought year** — precipitation is minimal after February, and the observed peaks before the gap are small. The ranking as "dry" is likely correct even with truncated data, but **model evaluation on WY 2009 is compromised** since we're missing ~7 months of flow for 3 basins.

In [ ]:
plot_flow_precip(basin_data, SPLITS, save_path=PLOT_DIR / 'flow_precip_full_record.png')

### Plot the extremeness bar charts

In [ ]:
for basin in BASINS:
    plot_basin_extremeness(basin, df_sample, splits=SPLITS, save_path=PLOT_DIR / f'extremeness_bars_{basin.lower().replace(" ", "_")}.png')

### Plot extremeness + precip + flow

In [ ]:
for basin in BASINS:
    plot_basin_flow_precip_extremeness(basin, basin_data, df_sample, splits=SPLITS, save_path=PLOT_DIR / f'flow_precip_extremeness_{basin.lower().replace(" ", "_")}.png')

In [ ]:
candidates = [1995, 2006, 2001, 2007, 2009]

comp_rows = []
for wy in candidates:
    row = {'water_year': wy}
    row['mean_rank'] = float(ranks.loc[wy, 'mean_rank'])
    row['mean_exceedance'] = round(float(df_sample.loc[wy, BASINS].mean()), 3)

    for basin in BASINS:
        bdf = basin_data[basin]
        wy_flow = bdf.loc[bdf['water_year'] == wy, 'flow']
        n_total = len(wy_flow)
        n_obs = wy_flow.notna().sum()
        row[f'{basin}'] = f"{n_obs}/{n_total} ({n_obs/n_total*100:.0f}%)"

    comp_rows.append(row)

df_comp = pd.DataFrame(comp_rows).set_index('water_year')
print("Extreme Year Data Completeness")
print("  1995: DROPPED (90-day lookback corruption)")
print("  2009: Val only (peak captured, 59% missing after Feb 26)")
df_comp

## Final Extreme Year Sequences for USACE

Two extreme holdout sequences across WY1996–2009 (14 water years, WY1995 dropped). Both hold out 2 wet + 2 dry extremes in test. No separate validation set — cross-validation handles hyperparameter selection internally.

### Decision: Drop WY1995

WY1995 has 226 days NaN across 3/4 basins (Oct–Dec 1994, Feb–Jun 1995). The Jan 1995 peak is observed, but the **90-day LSTM lookback window** for peak prediction includes corrupted antecedent conditions (Oct–Dec 1994 NaN).

Options considered and rejected:
1. **Masked-Loss**: Forward pass still blind to antecedent conditions — LSTM hidden state is corrupted.
2. **HMS Imputation**: Introduces HMS bias into the target variable, contaminates LSTM-vs-HMS comparison.
3. **Drop entirely** (**adopted**): Cleanest and most defensible. WY2006 serves as the clean extreme wet benchmark.

### Sequence A — 1997 in Test

| Split | Water Years | Duration |
|:---|:---|:---|
| **Train (with CV)** | 1996, 1998, 1999, 2000, 2002, 2003, 2004, 2005, 2008, 2009 | 10 yr |
| **Test** | 1997, 2001, 2006, 2007 | 4 yr |

**Training exceedance range**: 0.164–0.834 (includes 2003, ranked #3 wettest)

**Test years** (2 wet + 2 dry):
- **2006** (exc. 0.065): Extreme wet — AR-driven flood, Weak La Niña + Warm PDO
- **1997** (exc. 0.174): Very wet — catastrophic El Niño flood, Warm PDO in-phase
- **2001** (exc. 0.932): Extreme dry — driest year in record across all 4 basins
- **2007** (exc. 0.820): Severe drought — Cool PDO + La Niña persistent ridge

### Climate Context (WY1996–2009) - from Gemini research... idk

| WY | DWR SVI | ENSO (ONI) | PDO | PDSI | Pearson-III Exc. |
|----|---------|------------|-----|------|:---:|
| 1996 | Wet | Weak La Niña | Warm (+) | Severely Wet (W2) | 0.617 |
| 1997 | Wet | Neutral → El Niño | Warm (+) | Extremely Wet (W3) | 0.174 |
| 1998 | Wet | Very Strong El Niño | Warm (+) | Exceptionally Wet (W4) | 0.235 |
| 1999 | Wet | Strong La Niña | Cool (−) | Moderately Wet (W1) | 0.572 |
| 2000 | Above Normal | Strong La Niña | Cool (−) | Normal/Dry (D0) | 0.712 |
| 2001 | **Dry** | Neutral | Cool (−) | Moderate Drought (D1) | **0.932** |
| 2002 | Dry | Neutral | Cool (−) | Moderate Drought (D1) | 0.678 |
| 2003 | Above Normal | Mod El Niño | Warm (+) | Moderately Wet (W1) | 0.164 |
| 2004 | Below Normal | Neutral | Warm (+) | Abnormally Dry (D0) | 0.396 |
| 2005 | Below Normal | Weak El Niño | Warm (+) | Normal/Wet (W0) | 0.796 |
| 2006 | **Wet** | Weak La Niña | Warm (+) | Severely Wet (W2) | **0.065** |
| 2007 | **Dry** | Weak El Niño | Cool (−) | Severe Drought (D2) | **0.820** |
| 2008 | Critical | Strong La Niña | Cool (−) | Extreme Drought (D3) | 0.521 |
| 2009 | Dry | Weak La Niña | Cool (−) | Severe Drought (D2) | 0.834 |

**Four Climate Regimes**:
1. **Extreme Wet (1996–1998)**: Warm PDO + El Niño in-phase → amplified AR moisture transport
2. **Cold Transition (1999–2002)**: Cool PDO + La Niña → storm track suppression, drought onset
3. **Mixed Recovery (2003–2006)**: Warm PDO return + variable ENSO → 2006 AR flood interruption
4. **Severe Drought (2007–2009)**: Cool PDO + La Niña → persistent high-pressure ridge

### USACE Delivery Summary

**Two extreme holdout sequences** for WY1996–2009 (14 water years). CV handles hyperparameter selection — no separate validation set.

| | Sequence A (1997 in test) | Sequence B (2003 in test) |
|:---|:---|:---|
| **Train (with CV)** | 1996, 1998–2000, 2002–05, 2008–09 (10yr) | 1996–2000, 2002, 2004–05, 2008–09 (10yr) |
| **Test** | 1997, 2001, 2006, 2007 (4yr) | 2001, 2003, 2006, 2007 (4yr) |
| **Train exc.** | 0.164–0.834 (includes 2003) | 0.174–0.834 (includes 1997) |
| **Ablation** | Trains on flashy AR (2003), tests on El Niño (1997) | Trains on El Niño (1997), tests on flashy AR (2003) |

**Test years (shared)**: 2006 (extreme wet), 2001 (extreme dry), 2007 (severe drought)
**Test years (swapped)**: 1997 (Seq A) ↔ 2003 (Seq B)

**Key decisions**:
- **WY1995 dropped**: 226-day NaN corrupts 90-day lookback; no imputation (avoids HMS bias contamination)
- **No separate val**: CV folds handle hyperparam selection internally; maximizes training data
- **Rankings**: Pearson Type III (LP3), sample skew, 7-day average peak flows (Bulletin 17C)

In [ ]:
from matplotlib.patches import Patch

fig, axes = plt.subplots(3, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [1, 1, 2.5]})

split_colors = {'train': '#4A90D9', 'test': '#D0021B'}
all_wys = list(range(1996, 2010))
mean_exc = {wy: round(float(df_sample.loc[wy, BASINS].mean()), 3) for wy in all_wys}

sequences = {
    'Sequence A\n(1997 in test)': seq_a,
    'Sequence B\n(2003 in test)': seq_b,
}

# --- Top 2 rows: sequence timelines ---
for ax_idx, (name, seq) in enumerate(sequences.items()):
    ax = axes[ax_idx]
    for wy in all_wys:
        split = next(s for s, yrs in seq.items() if wy in yrs)
        ax.barh(0, 1, left=wy - 0.5, color=split_colors[split], edgecolor='white', linewidth=2, height=0.7)
        ax.text(wy, 0, str(wy), ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    # Gray out 1995
    ax.barh(0, 1, left=1994.5, color='#CCCCCC', edgecolor='white', linewidth=2, height=0.7)
    ax.text(1995, 0, '1995', ha='center', va='center', fontsize=7, color='#666666', style='italic')

    ax.set_ylabel(name, fontsize=9, fontweight='bold', rotation=0, labelpad=100, va='center')
    ax.set_xlim(1994.3, 2009.7)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)

# Legend on top row
legend_elements = [
    Patch(facecolor='#CCCCCC', edgecolor='white', label='Dropped (1995)'),
    Patch(facecolor=split_colors['train'], edgecolor='white', label='Train (with CV)'),
    Patch(facecolor=split_colors['test'], edgecolor='white', label='Test'),
]
axes[0].legend(handles=legend_elements, loc='upper right', ncol=3, fontsize=8, framealpha=0.9)

# --- Bottom: exceedance bars colored by Seq A ---
ax = axes[2]
bar_x = [1995] + all_wys
bar_exc = [0.062] + [mean_exc[wy] for wy in all_wys]

bar_colors = ['#CCCCCC']
for wy in all_wys:
    split = next(s for s, yrs in seq_a.items() if wy in yrs)
    bar_colors.append(split_colors[split])

bars = ax.bar(bar_x, bar_exc, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.8)

for x, exc in zip(bar_x, bar_exc):
    ax.text(x, exc + 0.015, f'{exc:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_ylabel('Mean Pearson-III Exceedance\n(lower = more extreme wet)', fontsize=9)
ax.set_xticks(bar_x)
ax.set_xticklabels([str(y) for y in bar_x], fontsize=9)
ax.set_xlabel('Water Year', fontsize=10)
ax.set_ylim(0, 1.08)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.2, linewidth=0.8)
ax.text(2009.5, 0.51, 'median', fontsize=7, alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)

# Highlight the swap between A and B
ax.annotate('A: test / B: train', xy=(1997, mean_exc[1997]), xytext=(1997, 0.38),
            fontsize=7, ha='center', color='#D0021B', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#D0021B', lw=1))
ax.annotate('A: train / B: test', xy=(2003, mean_exc[2003]), xytext=(2003, 0.38),
            fontsize=7, ha='center', color='#4A90D9', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#4A90D9', lw=1))

fig.suptitle('Proposed Extreme Year Sequences (WY1996–2009, no separate val)', fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'proposed_sequences_AB.png', dpi=150, bbox_inches='tight')
plt.show()